# Evaluation

A few good-looking answers do not tell us whether an agent works.

In this notebook, we will:

- see where the known answers come from;
- score one prediction field by field;
- run the same test over several tickets;
- compare a baseline with a regression; and
- inspect one failed run.

In [ ]:
import copy
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Gold answers come from rules

A gold answer is the result we expect. Here it is not a human opinion. The shop's rules calculate it from the ticket, order, and customer.

In [ ]:
from shoplab import rules, world

tickets = world.load_tickets()
orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

ticket = tickets["train"][1]
calculated = rules.decide(
    ticket,
    orders[ticket["order_id"]],
    customers[ticket["customer_id"]],
)

print("stored:    ", ticket["gold"])
print("calculated:", calculated)

## 2. Score one prediction

We check three fields:

- decision;
- policy_id; and
- refund_usd.

The amount is correct when it is within one cent. If the gold amount is null, the prediction must also be null.

In [ ]:
def simple_score(prediction, gold):
    if not isinstance(prediction, dict):
        return {"decision": False, "policy": False, "amount": False, "exact": False}

    decision_ok = prediction.get("decision") == gold["decision"]
    policy_ok = prediction.get("policy_id") == gold["policy_id"]

    predicted_amount = prediction.get("refund_usd")
    gold_amount = gold["refund_usd"]
    if predicted_amount is None or gold_amount is None:
        amount_ok = predicted_amount is None and gold_amount is None
    elif isinstance(predicted_amount, (int, float)) and not isinstance(predicted_amount, bool):
        amount_ok = abs(predicted_amount - gold_amount) <= 0.01
    else:
        amount_ok = False

    return {
        "decision": decision_ok,
        "policy": policy_ok,
        "amount": amount_ok,
        "exact": decision_ok and policy_ok and amount_ok,
    }

In [ ]:
gold = ticket["gold"]
close_amount = {**gold, "refund_usd": gold["refund_usd"] + 0.005}
wrong_policy = {**gold, "policy_id": "made-up-policy"}

print("correct:", simple_score(gold, gold))
print("within a cent:", simple_score(close_amount, gold))
print("wrong policy:", simple_score(wrong_policy, gold))

The course version is shoplab.evals.score_ticket. It uses longer field names but follows the same rules.

In [ ]:
from shoplab.evals import score_ticket

score_ticket(wrong_policy, gold)

## 3. Turn one score into an evaluation

An evaluation runs the same function on many tickets and averages the results.

In [ ]:
def evaluate_small(run_one, items):
    rows = []
    for item in items:
        prediction = run_one(item)
        rows.append(simple_score(prediction, item["gold"]))

    return sum(row["exact"] for row in rows) / len(rows)

print("perfect example:", evaluate_small(lambda item: item["gold"], tickets["dev"][:3]))

Now we test the tool-using agent from chapter 02. We use three dev tickets to keep this lesson quick. Change the slice to all dev tickets for a fuller run.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

SYSTEM = (
    "Handle this Larkspur return. Look up the order, customer, and policy. "
    "Then call finish with decision, policy_id, and refund_usd."
)

def run_ticket(item):
    task = json.dumps({key: value for key, value in item.items() if key != "gold"})
    return run_agent(
        task,
        standard_tools(),
        model=MODEL,
        system=SYSTEM,
        max_steps=8,
    ).answer

In [ ]:
from shoplab.evals import evaluate

sample = tickets["dev"][:3]
baseline = evaluate(run_ticket, sample, name="simple-baseline")

The result reports decision, policy, amount, and exact accuracy. It also records the model cost used during the evaluation.

## 4. Catch a regression

A regression is a change that makes an earlier result worse.

We will reuse the baseline predictions and remove their refund amounts. This creates a clear bug without paying for another model run.

In [ ]:
baseline_predictions = {
    row["ticket_id"]: row["pred"]
    for row in baseline["rows"]
}

def missing_amount(item):
    prediction = baseline_predictions[item["ticket_id"]]
    if not isinstance(prediction, dict):
        return prediction
    changed = copy.deepcopy(prediction)
    changed["refund_usd"] = None
    return changed

regressed = evaluate(missing_amount, sample, name="simple-missing-amount")

In [ ]:
from shoplab.evals import compare_runs

compare_runs("simple-baseline", "simple-missing-amount")

## 5. Read a failure, not only an average

An average tells us how often something failed. The saved row tells us what failed.

In [ ]:
rows = []
for row in regressed["rows"]:
    rows.append({
        "ticket_id": row["ticket_id"],
        "predicted": row["pred"],
        "gold": row["gold"],
        "exact": row["scores"]["exact"],
    })

pd.DataFrame(rows)

The on_step hook lets us replay an agent call and see which tools it chose.

In [ ]:
failed_ticket = sample[0]
failed_task = json.dumps({key: value for key, value in failed_ticket.items() if key != "gold"})

def show_step(step, message):
    calls = [call.function.name for call in message.tool_calls or []]
    print("step", step, calls or [message.content])

replay = run_agent(
    failed_task,
    standard_tools(),
    model=MODEL,
    system=SYSTEM,
    max_steps=8,
    on_step=show_step,
)

print("prediction:", replay.answer)
print("known:", failed_ticket["gold"])

## What to remember

1. Gold answers should come from a clear source of truth.
2. Score important fields separately.
3. Exact accuracy requires every field to agree.
4. Evaluate the same function over a fixed ticket set.
5. Save named runs so changes can be compared.
6. Read failed rows and transcripts to learn why the score changed.

## Small practice steps

1. Score a refund amount that is two cents too high.
2. Run the baseline on one more dev ticket.
3. Create a regression that changes policy_id instead of refund_usd.